In [3]:
# ======================================
# 🏥 HEALTHCARE RESEARCH ASSISTANT (GROQ)
# ======================================

# --------------------------------------
# STEP 1: Install Libraries
# --------------------------------------
!pip install groq gradio huggingface


# --------------------------------------
# STEP 2: Setup Groq API
# --------------------------------------
import os
import time
import concurrent.futures
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.environ['GROQ_API_KEY']

client = Groq(api_key=groq_api_key)


# --------------------------------------
# STEP 3: Simulated Medical Data Sources
# --------------------------------------

# 📰 Medical News
def fetch_medical_news(topic):
    time.sleep(1)
    return [
        f"New breakthrough in {topic} treatment using AI-assisted drugs",
        f"Global study shows improved outcomes in {topic} patients",
        f"WHO releases updated guidelines for {topic}"
    ]


# 🧪 Clinical Trials
def fetch_clinical_trials(topic):
    time.sleep(1)
    return [
        f"Phase 3 trial shows 65% success rate for new {topic} drug",
        f"Combination therapy improves survival rate in {topic}",
        f"New immunotherapy shows promising results in early trials"
    ]


# 🏢 Pharma Company Profiles
def fetch_pharma_profiles(topic):
    time.sleep(1)
    return {
        "companies": [
            f"Pfizer working on {topic} vaccines",
            f"Moderna developing mRNA-based {topic} treatments",
            f"Novartis investing heavily in {topic} research"
        ]
    }


# --------------------------------------
# STEP 4: Parallel Data Collection
# --------------------------------------
def gather_medical_data(topic):

    with concurrent.futures.ThreadPoolExecutor() as executor:
        news_future = executor.submit(fetch_medical_news, topic)
        trials_future = executor.submit(fetch_clinical_trials, topic)
        pharma_future = executor.submit(fetch_pharma_profiles, topic)

        news = news_future.result()
        trials = trials_future.result()
        pharma = pharma_future.result()

    return news, trials, pharma


# --------------------------------------
# STEP 5: LLM Analysis + Report
# --------------------------------------
def generate_medical_report(topic, news, trials, pharma):

    try:
        prompt = f"""
        You are a medical research assistant.

        Generate a professional healthcare research report.

        Topic: {topic}

        Medical News:
        {news}

        Clinical Trials:
        {trials}

        Pharma Companies:
        {pharma}

        Include:
        - Overview of topic
        - Key Research Findings
        - Clinical Insights
        - Industry Trends
        - Risks & Limitations
        - Conclusion

        Keep it clear and suitable for doctors/researchers.
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0.4,
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        print("LLM Error:", e)
        return "⚠️ Failed to generate medical report"


# --------------------------------------
# STEP 6: Main Agent
# --------------------------------------
def healthcare_agent(message, history):

    topic = message.strip()

    # Step 1: Parallel data collection
    news, trials, pharma = gather_medical_data(topic)

    # Step 2: LLM analysis
    report = generate_medical_report(topic, news, trials, pharma)

    history.append((message, report))
    return "", history


# --------------------------------------
# STEP 7: Gradio UI
# --------------------------------------
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Healthcare Research Assistant")

    chatbot = gr.Chatbot(height=500)

    msg = gr.Textbox(label="Enter topic (e.g., diabetes, cancer drugs)")

    state = gr.State([])

    msg.submit(
        healthcare_agent,
        inputs=[msg, state],
        outputs=[msg, chatbot]
    )


# --------------------------------------
# STEP 8: Launch
# --------------------------------------
demo.launch(share=True)

Defaulting to user installation because normal site-packages is not writeable


ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (C:\Users\sk165\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\__init__.py)